<a href="https://colab.research.google.com/github/martirossi/AppliedML2026_mr/blob/main/regression_catboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# ============================================
# CatBoost Regressor + SHAP Feature Selection + Final Test Predictions
# ============================================

import numpy as np
import pandas as pd
import shap

!pip install catboost
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ------------------------------------------------
# 1. Load training data (electrons only)
# ------------------------------------------------
train_path = "/content/drive/MyDrive/Colab Notebooks/AppML_InitialProject_train.csv"
data = pd.read_csv(train_path)

# Keep only electrons
data = data[data["p_Truth_isElectron"] == 1]

target_col = "p_Truth_Energy"
y = data[target_col].values

# Detect ID column
id_col = None
for col in data.columns:
    if col.lower() in ["eventid", "id"]:
        id_col = col

# Feature columns
feature_cols = [c for c in data.columns if c not in [target_col, id_col]]
X_full = data[feature_cols].values

In [7]:
# ------------------------------------------------
# 2. Temporary CatBoost model for SHAP ranking
# ------------------------------------------------
temp_model = CatBoostRegressor(
    iterations=3000,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    od_type="Iter",   # early stopping
    od_wait=50,
    verbose=False
)

temp_model.fit(X_full, y)

# SHAP
explainer = shap.TreeExplainer(temp_model)
shap_values = explainer.shap_values(X_full)

shap_importance = np.abs(shap_values).mean(axis=0)

shap_ranking = pd.DataFrame({
    "Feature": feature_cols,
    "SHAP_Importance": shap_importance
}).sort_values(by="SHAP_Importance", ascending=False)

top_20_features = shap_ranking["Feature"].head(20).tolist()

print("Selected 20 features using SHAP:")
for f in top_20_features:
    print(f)

Selected 20 features using SHAP:
pX_ecore
p_pt_track
pX_e233
pX_E3x5_Lr1
pX_maxEcell_energy
pX_E7x11_Lr1
p_sigmad0
pX_E_Lr1_MedG
pX_MultiLepton
pX_E_Lr2_HiG
p_etcone20
pX_topoetcone20
pX_E3x5_Lr2
p_ptPU30
pX_E_Lr2_MedG
p_ptcone40
pX_deltaPhiFromLastMeasurement
pX_E5x7_Lr1
pX_topoetcone40ptCorrection
pX_etcone20


In [8]:
# ------------------------------------------------
# 3. Prepare data with selected features
# ------------------------------------------------
X = data[top_20_features].values


In [9]:
# ------------------------------------------------
# 4. 5-fold CV with CatBoost + early stopping
# ------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
for train_idx, valid_idx in kf.split(X, y):
    print(f"\n===== Fold {fold} =====")

    X_train, X_valid = X[train_idx], X[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = CatBoostRegressor(
        iterations=5000,
        depth=6,
        learning_rate=0.05,
        loss_function="RMSE",
        random_seed=42,
        od_type="Iter",
        od_wait=50,
        verbose=False
    )

    model.fit(X_train, y_train, eval_set=(X_valid, y_valid))

    preds = model.predict(X_valid)

    mse = mean_squared_error(y_valid, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_valid, preds)
    relmad = np.mean(np.abs((preds - y_valid) / y_valid))

    print(f"Fold MSE   : {mse:.4f}")
    print(f"Fold RMSE  : {rmse:.4f} GeV")
    print(f"Fold MAE   : {mae:.4f} GeV")
    print(f"Fold RelMAD: {relmad:.4f}")

    fold += 1


===== Fold 1 =====
Fold MSE   : 297051904.5949
Fold RMSE  : 17235.1938 GeV
Fold MAE   : 8086.1751 GeV
Fold RelMAD: 0.3018

===== Fold 2 =====
Fold MSE   : 300713154.6281
Fold RMSE  : 17341.0829 GeV
Fold MAE   : 7830.1317 GeV
Fold RelMAD: 0.2760

===== Fold 3 =====
Fold MSE   : 281854921.6175
Fold RMSE  : 16788.5354 GeV
Fold MAE   : 8153.8278 GeV
Fold RelMAD: 0.3028

===== Fold 4 =====
Fold MSE   : 269631617.2928
Fold RMSE  : 16420.4634 GeV
Fold MAE   : 7838.3377 GeV
Fold RelMAD: 0.2670

===== Fold 5 =====
Fold MSE   : 292964713.9399
Fold RMSE  : 17116.2120 GeV
Fold MAE   : 8074.5183 GeV
Fold RelMAD: 0.2917


In [10]:
# ------------------------------------------------
# 5. Train final CatBoost model on ALL data
# ------------------------------------------------
final_model = CatBoostRegressor(
    iterations=5000,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    od_type="Iter",
    od_wait=50,
    verbose=False
)

final_model.fit(X, y)


CatBoostRegressor(depth=6, iterations=5000, learning_rate=0.05, loss_function='RMSE', od_type='Iter', od_wait=50, random_seed=42, verbose=False)

In [11]:
# ------------------------------------------------
# 6. Load test data and predict
# ------------------------------------------------
test_path = "/content/drive/MyDrive/Colab Notebooks/AppML_InitialProject_test_regression.csv"
test_data = pd.read_csv(test_path)

# Detect ID column
test_id_col = None
for col in test_data.columns:
    if col.lower() in ["eventid", "id"]:
        test_id_col = col
        break

if test_id_col is not None:
    test_ids = test_data[test_id_col].values
else:
    test_ids = np.arange(len(test_data))

X_test = test_data[top_20_features].values
test_preds = final_model.predict(X_test)

In [12]:
# ------------------------------------------------
# 7. Save required files (CatBoost version)
# ------------------------------------------------

submission_df = pd.DataFrame({
    "Index": np.arange(len(test_preds)),
    "Predicted_Energy": test_preds
})
submission_df.to_csv("Regression_MartinaRossi_CatBoost.csv", index=False)

varlist_df = pd.DataFrame({"FeatureName": top_20_features})
varlist_df.to_csv("Regression_MartinaRossi_CatBoost_VariableList.csv", index=False)

print("\nSaved files:")
print(" - Regression_MartinaRossi_CatBoost.csv")
print(" - Regression_MartinaRossi_CatBoost_VariableList.csv")



Saved files:
 - Regression_MartinaRossi_CatBoost.csv
 - Regression_MartinaRossi_CatBoost_VariableList.csv


In [13]:
final_model.tree_count_

5000